<a href="https://www.kaggle.com/code/ashrafulislamtanzil/data-mining?scriptVersionId=340212040" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# xrVLM — Stage 1 CNN: Chest X-ray Multi-Label Diagnosis (EfficientNet-B4)

This notebook implements **only the CNN stage** (Stage 1) of the xrVLM two-stage
chest X-ray diagnostic pipeline — no VLM / CheXagent involved. That keeps things
self-contained and avoids the Stage-2 VLM loading problems described in
`fallback_problem.md` of the project (`AutoProcessor` not available for the
text-only CheXagent checkpoint on Kaggle).

**Pipeline covered here:**
1. Load & clean the NIH ChestX-ray14 sample metadata + images
2. Exploratory data analysis
3. Patient-level train / val / test split
4. `Dataset` / `DataLoader` with the project's transform policy (no h-flip — preserves cardiac laterality)
5. EfficientNet-B4 CNN with a multi-label head + temperature-scaling calibration layer
6. Two-phase training (frozen-backbone warm-up → full fine-tune)
7. Post-hoc temperature calibration
8. Test-set evaluation (per-class AUROC, precision/recall/F1) + plots
9. Save checkpoint + results to `/kaggle/working/`

> **Update:** Stage-1 backbone swapped from ImageNet-pretrained EfficientNet-B4 to **torchxrayvision's DenseNet-121**, which is pretrained directly on chest X-rays (NIH + other CXR sources) instead of everyday photos. This gives the model a head start on lung/rib/fluid patterns before it ever sees your training images — usually the single biggest lever on a small dataset like this one.

> **Update 2:** Now auto-detects **either** the small "sample" dataset (`sample_labels.csv`, 5,606 images) **or** the full NIH ChestX-ray14 dataset (`Data_Entry_2017.csv`, 112,120 images) — whichever you attach. Epoch counts were trimmed for the full dataset since each epoch now contains ~20x more training steps; raise them back up if the loss curve is still clearly improving when training ends.

> **Update 3:** Backbone default → `densenet121-res224-all` (pretrained on NIH+CheXpert+MIMIC+PadChest+RSNA+OpenI combined, broader than NIH-only). Fixed a checkpoint bug where the fine-tune phase's val loss kept rising and the saved model silently stayed at the 2-epoch warmup state — checkpointing now selects on **val macro AUROC** instead of val loss, so a model that finetunes worse-on-loss-but-better-on-ranking still gets kept. Finetune LR lowered (1e-4→2e-5) and warmup extended (2→4 epochs) to stop the backbone forgetting its pretraining. Threshold tuning added at the end — fixes the low precision/F1 from a flat 0.5 cutoff.

**Before running on Kaggle:**
- Attach the NIH ChestX-ray "sample" dataset (or your own `sample_labels.csv` + PNGs) as a Kaggle Dataset input. The notebook locates `sample_labels.csv` and then **recursively scans the whole dataset folder for PNGs**, so it works no matter how deeply the images are nested (`images/`, `sample/images/`, etc.) or what the exact mount path is (e.g. `/kaggle/input/sample`, `/kaggle/input/datasets/organizations/nih-chest-xrays/sample`).
- Turn on a **GPU** accelerator (Settings → Accelerator → GPU T4 x2 or P100).
- The official Kaggle "nih-chest-xrays/sample" dataset has **5,606–5,608 images** with usable labels — enough to see real signal, though still far smaller than the full NIH ChestX-ray14 release (112,120 images). For maximum performance, swap in the full dataset (same file layout) — the code below does not need to change.


In [ ]:
# ── Install / import ────────────────────────────────────────────────────────
!pip install -q torchxrayvision

import os, glob, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.functional as TF
from PIL import Image, UnidentifiedImageError

import torchxrayvision as xrv
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
# Auto-detect the dataset root under /kaggle/input by locating the metadata
# CSV. Works with EITHER the small "sample" dataset (sample_labels.csv,
# 5,606 images) OR the full NIH ChestX-ray14 dataset (Data_Entry_2017.csv,
# 112,120 images, in 12 images_XXX/images/ subfolders) — whichever is attached.

def find_dataset_root(start="/kaggle/input"):
    for filename in ["Data_Entry_2017.csv", "Data_Entry_2017_v2020.csv", "sample_labels.csv"]:
        matches = glob.glob(os.path.join(start, "**", filename), recursive=True)
        if matches:
            return Path(matches[0]).parent, filename
    return None, None

DATA_ROOT, CSV_FILENAME = find_dataset_root()
if DATA_ROOT is None:
    # Fallback for local/offline runs — edit this path as needed.
    DATA_ROOT = Path("./sample_dataset")
    CSV_FILENAME = "sample_labels.csv"
    print(f"[WARN] Could not find a metadata CSV under /kaggle/input — "
          f"falling back to {DATA_ROOT}. Attach the dataset or fix this path.")
else:
    is_full = "Data_Entry" in CSV_FILENAME
    print(f"Dataset root: {DATA_ROOT}")
    print(f"Metadata file: {CSV_FILENAME}  ({'FULL 112k dataset' if is_full else 'sample dataset'})")

CSV_PATH = DATA_ROOT / CSV_FILENAME
# NOTE: images are located with a recursive scan in the cleaning step below —
# we deliberately do NOT hardcode a fixed subfolder, since the exact nesting
# depth varies (flat "images/", nested "sample/images/", or the full
# dataset's 12 "images_XXX/images/" folders). This is what makes the
# notebook find ALL images regardless of which dataset or mount layout.

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
CKPT_DIR = OUTPUT_DIR / "checkpoints"
FIGURES_DIR = OUTPUT_DIR / "figures"
for d in [OUTPUT_DIR, CKPT_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 14 NIH CXR-14 pathology labels ("No Finding" is implicit: all-zero row)
DISEASE_LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(DISEASE_LABELS)

IMAGE_SIZE = 224                 # fixed input size for torchxrayvision "res224" models

BACKBONE = "densenet121-res224-all"   # torchxrayvision — pretrained on NIH+CheXpert+MIMIC+
                                       # PadChest+RSNA+OpenI combined (broader than NIH-only)
DROPOUT = 0.4
TEMPERATURE_INIT = 1.5

BATCH_SIZE = 32
LR_WARMUP = 1e-3
LR_FINETUNE = 2e-5   # lowered — 1e-4 was causing the backbone to forget its
                     # chest X-ray pretraining during fine-tune (val loss rose every epoch)
# With the full 112k-image dataset each epoch already contains ~20x more
# training steps than the 5.6k sample, so fewer epochs are usually enough —
# these are a safer starting point for an ~8-12h Kaggle session. Watch the
# loss curve: raise these if val loss is still clearly dropping at the end.
EPOCHS_WARMUP = 4    # more warm-up — lets the head stabilize before backbone unfreezes
EPOCHS_FINETUNE = 6
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

# Colors (validated categorical / sequential palette — colorblind-safe)
C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"


## 1. Data Loading & Cleaning

The raw metadata CSV (`sample_labels.csv`) has **5,606** rows, but only a subset
of the referenced images are actually present on disk in this sample dataset
(~2,078). The most important cleaning step is therefore filtering the metadata
down to rows whose image file actually exists — training on missing files would
otherwise crash the `DataLoader`.

Cleaning steps applied below:
1. Drop duplicate `Image Index` rows.
2. Keep only rows whose image file exists on disk.
3. Parse `Patient Age` (`"060Y"` → `60`) and drop physiologically impossible ages (a known NIH CXR-14 data quirk — a handful of ages are recorded as >120).
4. Multi-hot encode the pipe-separated `Finding Labels` string into 14 binary disease columns.
5. Verify every remaining image actually opens (catches truncated/corrupt PNGs) and drop any that don't.


In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(f"Rows in CSV: {len(df_raw):,}")

# 1. Drop duplicate Image Index rows
df = df_raw.drop_duplicates(subset="Image Index").copy()
n_after_dedup = len(df)

# 2. Recursively index every PNG under the dataset root (handles any folder
#    depth — flat images/, nested sample/images/, etc.) and keep only rows
#    whose image file actually exists on disk. filename -> full Path.
image_index = {p.name: p for p in DATA_ROOT.rglob("*.png")}
print(f"Images found on disk (recursive scan of {DATA_ROOT}): {len(image_index):,}")
df = df[df["Image Index"].isin(image_index)].copy()
n_after_exists = len(df)

# 3. Clean Patient Age: "060Y" -> 60, drop impossible ages
def parse_age(v):
    try:
        return int(str(v).rstrip("Yy"))
    except ValueError:
        return np.nan

df["Age"] = df["Patient Age"].apply(parse_age)
bad_age = ~df["Age"].between(0, 110)
if bad_age.sum():
    print(f"Dropping {bad_age.sum()} rows with implausible age (e.g. NIH's known >120y entries)")
df = df[~bad_age].copy()
n_after_age = len(df)

# 4. Multi-hot encode the 14 disease labels
for label in DISEASE_LABELS:
    df[label] = df["Finding Labels"].str.contains(label, regex=False).astype(int)
df["No_Finding"] = (df[DISEASE_LABELS].sum(axis=1) == 0).astype(int)

# 5. Verify images actually open (catches corrupt/truncated files)
def is_readable(fname):
    try:
        with Image.open(image_index[fname]) as im:
            im.verify()
        return True
    except (UnidentifiedImageError, OSError):
        return False

readable_mask = df["Image Index"].apply(is_readable)
n_corrupt = (~readable_mask).sum()
if n_corrupt:
    print(f"Dropping {n_corrupt} unreadable/corrupt image files")
df = df[readable_mask].reset_index(drop=True)
n_final = len(df)

print("\n── Cleaning summary ──────────────────────────")
print(f"  raw rows                 : {len(df_raw):,}")
print(f"  after dedup              : {n_after_dedup:,}")
print(f"  after image-exists filter: {n_after_exists:,}")
print(f"  after age sanity filter  : {n_after_age:,}")
print(f"  after corrupt-image drop : {n_final:,}")
print(f"  unique patients          : {df['Patient ID'].nunique():,}")

df.head()


In [ ]:
print("Per-label positive counts (cleaned dataset):\n")
counts = df[DISEASE_LABELS + ["No_Finding"]].sum().sort_values(ascending=False)
print(counts.to_string())

rare = counts[(counts.index != "No_Finding") & (counts < 20)]
if len(rare):
    print(f"\n[NOTE] Very few positive examples for: {list(rare.index)}. "
          f"Per-class AUROC for these will be noisy on such a small sample — "
          f"expect this to improve a lot with the full NIH dataset.")


## 2. Exploratory Data Analysis

In [ ]:
# Class frequency — a single-series magnitude comparison, so one hue (sequential blue)
counts_sorted = df[DISEASE_LABELS].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(counts_sorted.index, counts_sorted.values, color=C_BLUE, height=0.65)
ax.set_xlabel("Number of positive images")
ax.set_title("Label frequency (cleaned dataset)")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "label_frequency.png", dpi=140)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Findings-per-image histogram (single series -> single hue)
n_findings = df[DISEASE_LABELS].sum(axis=1)
axes[0].hist(n_findings, bins=range(0, int(n_findings.max()) + 2), color=C_BLUE,
             align="left", rwidth=0.7)
axes[0].set_xlabel("Number of findings per image")
axes[0].set_ylabel("Image count")
axes[0].set_title("Findings per image")
axes[0].spines[["top", "right"]].set_visible(False)

# Gender split (2 categories -> categorical slots 1 & 2, validated pair)
gender_counts = df["Patient Gender"].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values, color=[C_BLUE, C_ORANGE], width=0.5)
axes[1].set_title("Patient gender")
axes[1].set_ylabel("Image count")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_overview.png", dpi=140)
plt.show()


In [ ]:
# Sample images with their labels
sample_rows = df.sample(n=min(8, len(df)), random_state=SEED)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    img = Image.open(image_index[row["Image Index"]]).convert("L")
    ax.imshow(img, cmap="gray")
    labels = [l for l in DISEASE_LABELS if row[l] == 1] or ["No Finding"]
    ax.set_title(", ".join(labels), fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sample_images.png", dpi=140)
plt.show()


## 3. Train / Val / Test Split

Split at the **patient level** (not image level) so the same patient's images
never leak across splits — the same policy used in the project's `dataset.py`.
No official NIH split-list files ship with this sample, so we do a random
70/15/15 split by patient.


In [ ]:
rng = np.random.default_rng(SEED)
patients = df["Patient ID"].unique()
rng.shuffle(patients)

n_patients = len(patients)
n_test = max(1, int(n_patients * TEST_FRACTION))
n_val = max(1, int(n_patients * VAL_FRACTION))

test_patients = set(patients[:n_test])
val_patients = set(patients[n_test:n_test + n_val])
train_patients = set(patients[n_test + n_val:])

train_df = df[df["Patient ID"].isin(train_patients)].reset_index(drop=True)
val_df = df[df["Patient ID"].isin(val_patients)].reset_index(drop=True)
test_df = df[df["Patient ID"].isin(test_patients)].reset_index(drop=True)

print(f"Patients   — train: {len(train_patients):,}  val: {len(val_patients):,}  test: {len(test_patients):,}")
print(f"Images     — train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

# Per-class pos_weight for BCEWithLogitsLoss (handles class imbalance)
pos = train_df[DISEASE_LABELS].sum(axis=0).values.astype(np.float32)
neg = len(train_df) - pos
pos_weight = torch.tensor(neg / np.maximum(pos, 1.0), dtype=torch.float32)
print("\npos_weight per class:")
for l, w in zip(DISEASE_LABELS, pos_weight.tolist()):
    print(f"  {l:<20s} {w:8.2f}")


## 4. Dataset & Transforms

Uses **torchxrayvision's** own preprocessing instead of ImageNet normalization:
convert to grayscale, scale pixel values into xrv's expected range, then a
center-crop + resize to 224×224 (the resolution the pretrained DenseNet-121
was trained at). Still **no horizontal flip**, since it would destroy
left/right laterality cues (e.g. cardiac silhouette position) that matter
clinically.


In [ ]:
xrv_resize = xrv.datasets.XRayResizer(IMAGE_SIZE)
xrv_crop = xrv.datasets.XRayCenterCrop()


def load_xrv_image(path, augment=False):
    img = Image.open(path).convert("L")              # grayscale
    img = np.array(img).astype(np.float32)
    img = xrv.datasets.normalize(img, 255)            # 8-bit -> xrv's expected range
    img = img[None, :, :]                             # add channel dim -> (1, H, W)
    img = xrv_crop(img)
    img = xrv_resize(img)

    if augment:
        # Mild rotation only — keeps the project's "no h-flip" policy
        # (a flip would reverse cardiac left/right laterality).
        angle = random.uniform(-10, 10)
        img_t = torch.from_numpy(img.copy())
        img_t = TF.rotate(img_t, angle)
        img = img_t.numpy()

    return torch.from_numpy(img.copy()).float()


class XRVChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_index, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_index = image_index          # filename -> full Path
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = load_xrv_image(self.image_index[row["Image Index"]], augment=self.augment)
        labels = torch.tensor(row[DISEASE_LABELS].values.astype(np.float32))
        return image, labels


train_ds = XRVChestXrayDataset(train_df, image_index, augment=True)
val_ds = XRVChestXrayDataset(val_df, image_index, augment=False)
test_ds = XRVChestXrayDataset(test_df, image_index, augment=False)

# Weighted sampler: up-weight images with fewer co-occurring findings (unchanged policy)
label_sums = train_ds.df[DISEASE_LABELS].sum(axis=1).values
sample_weights = np.where(label_sums > 0, 1.0 / np.maximum(label_sums, 1e-8), 1.0)
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float64),
    num_samples=len(train_ds), replacement=True,
)

NUM_WORKERS = 4   # more workers — full dataset streams a lot more images
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

images, labels = next(iter(train_loader))
print(f"Batch — images: {images.shape}  labels: {labels.shape}")


## 5. Model — torchxrayvision DenseNet-121 + Temperature Scaling

Backbone is `torchxrayvision`'s DenseNet-121, pretrained on chest X-rays
(NIH + other CXR sources) rather than ImageNet — so it starts already
knowing what lungs, ribs, and fluid look like. We keep the project's own
14-way classifier head on top, plus the same learned-temperature calibration
layer as before.


In [ ]:
class TemperatureScaler(nn.Module):
    """Post-hoc calibration: logits / T. T is tuned after training, on val data."""

    def __init__(self, init_temp: float = TEMPERATURE_INIT):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(init_temp, dtype=torch.float32)))

    @property
    def temperature(self):
        return self.log_temperature.exp()

    def forward(self, logits):
        return logits / self.temperature


class XRVClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=DROPOUT, xrv_weights=BACKBONE):
        super().__init__()
        # Chest-X-ray-pretrained DenseNet-121 (NOT ImageNet). We only use it as a
        # feature extractor via .features() — its own 18-way pretrained head is
        # discarded in favor of our own 14-way head below.
        self.backbone = xrv.models.DenseNet(weights=xrv_weights)
        self.feature_dim = 1024   # DenseNet-121 feature size
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.feature_dim, num_classes),
        )
        self.temp_scaler = TemperatureScaler()

    def forward_logits(self, x):
        feats = self.backbone.features2(x)    # (B, 1024) pooled feature vector (built-in xrv helper)
        return self.classifier(feats)

    def forward(self, x):
        return self.temp_scaler(self.forward_logits(x))


def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = ("classifier" in name) or ("temp_scaler" in name)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"[freeze_backbone] trainable: {trainable:,} / {total:,}")


def unfreeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = True
    print(f"[unfreeze_backbone] trainable: {sum(p.numel() for p in model.parameters()):,}")


model = XRVClassifier().to(device)
print(f"Feature dim: {model.feature_dim}   Initial temperature: {model.temp_scaler.temperature.item():.3f}")


## 6. Training — Two-Phase (frozen-backbone warm-up → full fine-tune)

**Phase 1 (warm-up):** freeze the EfficientNet backbone, train only the
classifier head — fast convergence without disturbing ImageNet features.
**Phase 2 (fine-tune):** unfreeze everything, train end-to-end at a lower
learning rate with `ReduceLROnPlateau`.

Loss is class-weighted `BCEWithLogitsLoss` (multi-label), matching the
imbalance in the label set (e.g. Hernia is ~35x rarer than Infiltration).


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)


@torch.no_grad()
def validate(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Returns (val_loss, val_macro_auroc). We checkpoint on AUROC, not loss —
    loss can keep rising from probability-scale drift even while the model's
    ability to RANK sick-vs-healthy (what AUROC measures) keeps improving."""
    model.eval()
    running_loss = 0.0
    all_probs, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        logits = model(images)
        running_loss += criterion(logits, labels).item()
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    probs = np.concatenate(all_probs)
    labels_np = np.concatenate(all_labels)
    aurocs = [roc_auc_score(labels_np[:, i], probs[:, i])
              for i in range(num_classes) if len(np.unique(labels_np[:, i])) > 1]
    macro_auroc = float(np.mean(aurocs)) if aurocs else float("nan")
    return running_loss / max(len(loader), 1), macro_auroc


def save_checkpoint(model, path):
    torch.save({
        "model_state_dict": model.state_dict(),
        "temperature": model.temp_scaler.temperature.item(),
    }, path)


In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
history = []  # (epoch_global, phase, train_loss, val_loss, val_auroc)
best_val_auroc = -1.0
epoch_global = 0
best_ckpt_path = CKPT_DIR / "best_model.pth"

# ── Phase 1: head warm-up (backbone frozen) ─────────────────────────────────
print("=" * 60, "\nPHASE 1 — Head warm-up (backbone frozen)\n", "=" * 60, sep="")
freeze_backbone(model)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                              lr=LR_WARMUP, weight_decay=1e-5)

for epoch in range(1, EPOCHS_WARMUP + 1):
    epoch_global += 1
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_auroc = validate(model, val_loader, criterion, device)
    print(f"  Epoch {epoch_global:>3} [warmup]  train: {train_loss:.4f}  val: {val_loss:.4f}  "
          f"val_auroc: {val_auroc:.4f}  ({time.time()-t0:.0f}s)")
    history.append((epoch_global, "warmup", train_loss, val_loss, val_auroc))
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        save_checkpoint(model, best_ckpt_path)
        print(f"    new best val macro AUROC: {best_val_auroc:.4f}")

# ── Phase 2: full fine-tune (backbone unfrozen) ─────────────────────────────
print("\n" + "=" * 60, "\nPHASE 2 — Full fine-tune (backbone unfrozen)\n", "=" * 60, sep="")
unfreeze_backbone(model)
optimizer = torch.optim.Adam(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

for epoch in range(1, EPOCHS_FINETUNE + 1):
    epoch_global += 1
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_auroc = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)
    lr = optimizer.param_groups[0]["lr"]
    print(f"  Epoch {epoch_global:>3} [finetune]  train: {train_loss:.4f}  val: {val_loss:.4f}  "
          f"val_auroc: {val_auroc:.4f}  lr: {lr:.1e}  ({time.time()-t0:.0f}s)")
    history.append((epoch_global, "finetune", train_loss, val_loss, val_auroc))
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        save_checkpoint(model, best_ckpt_path)
        print(f"    new best val macro AUROC: {best_val_auroc:.4f}")

print(f"\nTraining complete. Best val macro AUROC: {best_val_auroc:.4f}  ({best_ckpt_path})")


In [ ]:
hist_df = pd.DataFrame(history, columns=["epoch", "phase", "train_loss", "val_loss", "val_auroc"])
warmup_end = (hist_df["phase"] == "warmup").sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(hist_df["epoch"], hist_df["train_loss"], color=C_BLUE, linewidth=2, label="Train loss")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], color=C_ORANGE, linewidth=2, label="Val loss")
axes[0].axvline(warmup_end + 0.5, color="#c3c2b7", linestyle="--", linewidth=1)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE loss"); axes[0].set_title("Loss")
axes[0].legend(frameon=False); axes[0].spines[["top", "right"]].set_visible(False)
axes[0].grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0); axes[0].set_axisbelow(True)

axes[1].plot(hist_df["epoch"], hist_df["val_auroc"], color=C_AQUA, linewidth=2, label="Val macro AUROC")
axes[1].axvline(warmup_end + 0.5, color="#c3c2b7", linestyle="--", linewidth=1)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro AUROC"); axes[1].set_title("Val AUROC (checkpoint metric)")
axes[1].legend(frameon=False); axes[1].spines[["top", "right"]].set_visible(False)
axes[1].grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0); axes[1].set_axisbelow(True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves.png", dpi=140)
plt.show()


## 7. Post-hoc Temperature Calibration

Reload the best checkpoint, freeze everything except the single temperature
parameter, and tune it on the validation set (minimizing BCE) so that
`sigmoid(logits / T)` gives better-calibrated probabilities than the raw
logits — useful downstream if you later add a confidence-based routing rule.


In [ ]:
def tune_temperature(model, loader, device, lr=0.01, max_iter=100):
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    model.temp_scaler.log_temperature.requires_grad = True

    all_logits, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            all_logits.append(model.forward_logits(images))
            all_labels.append(labels.to(device))
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.LBFGS([model.temp_scaler.log_temperature], lr=lr, max_iter=max_iter)

    def _closure():
        optimizer.zero_grad()
        loss = criterion(model.temp_scaler(all_logits), all_labels)
        loss.backward()
        return loss

    optimizer.step(_closure)
    for p in model.parameters():
        p.requires_grad = True
    return model.temp_scaler.temperature.item()


ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

final_T = tune_temperature(model, val_loader, device)
print(f"Calibrated temperature: {final_T:.4f}")

calibrated_ckpt_path = CKPT_DIR / "best_model_calibrated.pth"
save_checkpoint(model, calibrated_ckpt_path)
print(f"Saved calibrated checkpoint: {calibrated_ckpt_path}")


## 8. Evaluation on the Held-Out Test Set

In [ ]:
@torch.no_grad()
def predict_all(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for images, labels in loader:
        images = images.to(device)
        probs = torch.sigmoid(model(images)).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


test_probs, test_labels = predict_all(model, test_loader, device)
test_preds = (test_probs >= 0.5).astype(int)

# Per-class AUROC — skip classes with < 2 positive (or < 2 negative) examples
# in the test split, since AUROC is undefined for a single-class target.
auroc_per_class = {}
for i, label in enumerate(DISEASE_LABELS):
    y_true = test_labels[:, i]
    if len(np.unique(y_true)) < 2:
        auroc_per_class[label] = np.nan
        continue
    auroc_per_class[label] = roc_auc_score(y_true, test_probs[:, i])

auroc_series = pd.Series(auroc_per_class).sort_values(ascending=False)
print("Per-class AUROC (test set):\n")
print(auroc_series.round(3).to_string())

macro_auroc = auroc_series.dropna().mean()
print(f"\nMacro-average AUROC (over classes with both pos+neg examples in test): {macro_auroc:.3f}")

print("\nClassification report @ threshold 0.5:\n")
print(classification_report(test_labels, test_preds, target_names=DISEASE_LABELS,
                             zero_division=0))


### 8b. Per-Class Threshold Tuning

The 0.5 cutoff above is the same for all 14 diseases, but each class's predicted-probability distribution sits in a different range (driven partly by each class's `pos_weight`). Tuning a separate threshold per class — chosen on the **validation** set, applied to the **test** set — fixes the low precision without needing to retrain anything.


In [ ]:
from sklearn.metrics import f1_score

@torch.no_grad()
def get_probs(model, loader, device):
    model.eval()
    probs, labels = [], []
    for images, y in loader:
        images = images.to(device)
        probs.append(torch.sigmoid(model(images)).cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(probs), np.concatenate(labels)

val_probs, val_labels = get_probs(model, val_loader, device)

best_thresholds = np.full(NUM_CLASSES, 0.5)
for i in range(NUM_CLASSES):
    y_true = val_labels[:, i]
    if len(np.unique(y_true)) < 2:
        continue
    candidates = np.arange(0.05, 0.95, 0.02)
    scores = [f1_score(y_true, (val_probs[:, i] >= t).astype(int), zero_division=0)
              for t in candidates]
    best_thresholds[i] = candidates[int(np.argmax(scores))]

print("Tuned per-class thresholds (val set):\n")
for l, t in zip(DISEASE_LABELS, best_thresholds):
    print(f"  {l:<20s} {t:.2f}")

# Re-score the TEST set with tuned thresholds instead of a flat 0.5
test_preds_tuned = (test_probs >= best_thresholds[None, :]).astype(int)
print("\nClassification report — tuned thresholds (test set):\n")
print(classification_report(test_labels, test_preds_tuned, target_names=DISEASE_LABELS,
                             zero_division=0))

from sklearn.metrics import hamming_loss, accuracy_score

# Per-class accuracy (tuned thresholds)
per_class_acc = (test_preds_tuned == test_labels).mean(axis=0)
acc_series = pd.Series(per_class_acc, index=DISEASE_LABELS).sort_values(ascending=False)
print("Per-class accuracy (tuned thresholds):\n")
print(acc_series.round(3).to_string())

# Two dataset-level accuracy numbers — they answer different questions:
hamming_acc = 1 - hamming_loss(test_labels, test_preds_tuned)      # avg correctness per label
subset_acc = accuracy_score(test_labels, test_preds_tuned)         # ALL 14 labels correct at once

print(f"\nHamming accuracy (avg per-label correctness): {hamming_acc:.3f}")
print(f"Subset accuracy (all 14 labels exactly right):  {subset_acc:.3f}")
print("\n[NOTE] Hamming accuracy looks high mostly because most labels are")
print("       negative for most images \u2014 a model that predicts 'no disease'")
print("       everywhere would already score ~90%+ here. Macro AUROC and")
print("       per-class F1 above are what actually measure whether the model")
print("       is finding real disease cases, not just being right by default.")


In [ ]:
plot_series = auroc_series.dropna().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(plot_series.index, plot_series.values, color=C_BLUE, height=0.65)
ax.axvline(0.5, color="#c3c2b7", linestyle="--", linewidth=1)
ax.text(0.505, -0.6, "random (0.5)", fontsize=8, color="#898781")
ax.set_xlim(0, 1)
ax.set_xlabel("AUROC")
ax.set_title("Per-class AUROC — test set")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "auroc_per_class.png", dpi=140)
plt.show()


In [ ]:
# A handful of test predictions vs. ground truth
sample_idx = np.random.default_rng(SEED).choice(len(test_df), size=min(8, len(test_df)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for ax, idx in zip(axes.flat, sample_idx):
    row = test_df.iloc[idx]
    img = Image.open(image_index[row["Image Index"]]).convert("L")
    ax.imshow(img, cmap="gray")

    true_labels = [l for l in DISEASE_LABELS if row[l] == 1] or ["No Finding"]
    pred_labels = [DISEASE_LABELS[i] for i in range(NUM_CLASSES) if test_preds[idx, i] == 1] or ["No Finding"]
    top_i = int(np.argmax(test_probs[idx]))
    title = (f"true: {', '.join(true_labels)}\n"
             f"pred: {', '.join(pred_labels)}\n"
             f"top p({DISEASE_LABELS[top_i]})={test_probs[idx, top_i]:.2f}")
    ax.set_title(title, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_predictions.png", dpi=140)
plt.show()


## 9. Save Outputs

In [ ]:
results = {
    "backbone": BACKBONE,
    "num_classes": NUM_CLASSES,
    "dataset_size": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "epochs": {"warmup": EPOCHS_WARMUP, "finetune": EPOCHS_FINETUNE},
    "best_val_loss": float(best_val_loss),
    "calibrated_temperature": float(final_T),
    "macro_auroc_test": float(macro_auroc),
    "per_class_auroc_test": {k: (None if np.isnan(v) else float(v)) for k, v in auroc_per_class.items()},
    "tuned_thresholds": {l: float(t) for l, t in zip(DISEASE_LABELS, best_thresholds)},
    "per_class_accuracy_test": {l: float(a) for l, a in zip(DISEASE_LABELS, per_class_acc)},
    "hamming_accuracy_test": float(hamming_acc),
    "subset_accuracy_test": float(subset_acc),
}

with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))
print(f"\nCheckpoints : {CKPT_DIR}")
print(f"Figures     : {FIGURES_DIR}")
print(f"Results     : {OUTPUT_DIR / 'results.json'}")
